# Turtle Breakout 12 — the risk portfolio at a survivable base risk (TB 11 repaired)

Pre-registration: `discussions/Turtle_Breakout/014_claude_tb12_preregistration.md`. Code: `turtle_portfolio.py` 0.2.0.

TB 11 put 2% of the high-water mark on every trade across 30 coins in one account and most runs were ruined, so its tests compared ruin with ruin. TB 12 changes two things only:
- **sizing on current equity** (risk, caps and open risk measured on the last marked equity);
- **base risk per trade** $f \in \{0.25\%, 0.5\%, 1\%\}$, **primary 0.5%** ($\approx 2\%/\sqrt{16}$); $H$ and $C$ grids scale with $f$ ($H \in \{2,3,4,5,7.5,10,15\}\,f$, $C \in \{1,2,3,4\}\,f$, the same position counts as TB 11).
Rules, windows and test are TB 11's: discovery 2020–2023 picks $H$, $C$ (Calmar); evaluation 2024 → end: Calmar(rule) − Calmar(R0m), Holm over 6 per $f$. **The evaluation window was already seen in TB 11.**

Environment: `TB01_DATA_DIR` (default `BINANCE_UM_DIR\long`), `TB12_SYNTHETIC=1`.

In [ ]:
import os, sys, json, gc, time, subprocess, tempfile, warnings, platform
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_trade_features as tf, turtle_sizing as ts, turtle_reverse as tr, turtle_portfolio as tp

SYNTHETIC = os.getenv('TB12_SYNTHETIC', '0') == '1'
P = tb.Params(); RP = tr.RParams()
PP0 = replace(tp.PParams(), size_base='equity', status='PREREGISTERED_TB014')
FS = (0.0025, 0.005, 0.01); F_PRIMARY = 0.005
H_MULT = (2, 3, 4, 5, 7.5, 10, 15); C_MULT = (1, 2, 3, 4)
DISC0, DISC1 = tp.DISC; EVAL0 = tp.EVAL_START
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb12-')) / '12'; PP0 = replace(PP0, n_boot=300); H_MULT = (2, 5, 10); C_MULT = (1, 3)
    DISC0, DISC1, EVAL0 = pd.Timestamp('2021-07-01', tz='UTC'), pd.Timestamp('2023-07-01', tz='UTC'), pd.Timestamp('2023-07-01', tz='UTC')
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '12'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_portfolio.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
REG = {**tp.registration(PP0), 'f_values': FS, 'f_primary': F_PRIMARY, 'H_multiples_of_f': H_MULT, 'C_multiples_of_f': C_MULT,
       'size_base': 'equity', 'note': 'evaluation window already seen in TB 11'}
print(json.dumps(REG, indent=1, default=str))

## 1. Books (as TB 11)

In [ ]:
BOOKS = {'long': {}, 'long_rev': {}}; COV = []
if SYNTHETIC:
    BOOKS['long'] = tp.synthetic_books(10, 1100, seed=3, with_reverse=False)
    BOOKS['long_rev'] = tp.synthetic_books(10, 1100, seed=3, with_reverse=True)
    COV = pd.DataFrame([{'asset': a, 'longs': len(b.trades), 'reverses': int((BOOKS['long_rev'][a].trades.kind == 'reverse').sum())} for a, b in BOOKS['long'].items()])
else:
    for a in tf.UNIVERSE:
        t1 = time.time()
        pu = DATA / f'{a}USDT_um_1m.parquet'
        if not pu.exists():
            print(a, 'missing'); continue
        b, _ = tf.load_um(pu)
        fpth = DATA / 'basis' / 'funding' / f'{a}USDT_funding.parquet'
        f = tb.funding_events(pd.read_parquet(fpth)) if fpth.exists() else None
        if a == 'TON':
            b = b[b.index < tr.TON_CUTOFF]
        D = tb.prepare_asset(a, b, f, P)
        start = b.index[0] + pd.Timedelta(days=30)
        L0 = ts.trade_list(D, P, start, RP, with_reverse=False); L1 = ts.trade_list(D, P, start, RP, with_reverse=True)
        BOOKS['long'][a] = tp.make_book(D, L0, P); BOOKS['long_rev'][a] = tp.make_book(D, L1, P)
        COV.append({'asset': a, 'start': start, 'end': b.index[-1], 'longs': len(L0), 'reverses': int((L1.kind == 'reverse').sum())})
        print(f'{a}: {len(L0)} longs, {COV[-1]["reverses"]} reverses, {time.time() - t1:.0f}s')
        del b, D; gc.collect()
    COV = pd.DataFrame(COV)
display(COV)
RET = tp.returns_panel(BOOKS['long'])

## 2. Discovery, evaluation and tests for each f

In [ ]:
def run(variant, rule, t0, t1, pp, books=None, **kw):
    o = tp.simulate(books or BOOKS[variant], rule, pp, t0, t1, R=RET, **kw)
    Dd = tp.daily_curve(o['equity'], pp.start_equity)
    return {**tp.metrics(Dd, pp.start_equity), 'max_open_risk': o['max_open_risk'], 'mean_open_risk': o['mean_open_risk'],
            'max_gross_lev': o['max_gross_lev'], **o['stats']}, Dd, o

DISC_ALL, EVAL_ALL, TEST_ALL, SEL_ALL, CURVES, OUTS = [], [], [], {}, {}, {}
T0 = time.time()
for f_ in FS:
    pp = replace(PP0, f=f_)
    for v in ('long', 'long_rev'):
        rows = []
        for rule, kw in [('R0', {}), ('R2', {}), ('R4', {})] + [('R1', {'H': m * f_}) for m in H_MULT] + [('R3', {'C': m * f_}) for m in C_MULT]:
            m, _, _ = run(v, rule, DISC0, DISC1, pp, **kw)
            rows.append({'f': f_, 'variant': v, 'rule': rule, 'H': kw.get('H', np.nan), 'C': kw.get('C', np.nan), **m})
        g = pd.DataFrame(rows); DISC_ALL.append(g)
        sel = {'H': tp.select(g[g.rule == 'R1'], 'H'), 'C': tp.select(g[g.rule == 'R3'], 'C')}; SEL_ALL[(f_, v)] = sel
        for rule, kw in (('R0', {}), ('R1', {'H': sel['H']}), ('R2', {}), ('R3', {'C': sel['C']}), ('R4', {})):
            m, Dd, o = run(v, rule, EVAL0, None, pp, **kw)
            EVAL_ALL.append({'f': f_, 'variant': v, 'rule': rule, 'param': kw.get('H', kw.get('C', np.nan)), **m}); CURVES[(f_, v, rule)] = Dd; OUTS[(f_, v, rule)] = o
        r0 = OUTS[(f_, v, 'R0')]['mean_open_risk']
        for rule in ('R1', 'R2', 'R3'):
            fm = f_ * OUTS[(f_, v, rule)]['mean_open_risk'] / r0
            m, Dd, o = run(v, 'R0', EVAL0, None, replace(pp, f=fm))
            EVAL_ALL.append({'f': f_, 'variant': v, 'rule': f'R0m_for_{rule}', 'param': fm, **m}); CURVES[(f_, v, f'R0m_{rule}')] = Dd
            t = tp.calmar_diff(CURVES[(f_, v, rule)], Dd, pp, pp.seed + len(TEST_ALL))
            t0 = tp.calmar_diff(CURVES[(f_, v, rule)], CURVES[(f_, v, 'R0')], pp, pp.seed + 100 + len(TEST_ALL))
            TEST_ALL.append({'f': f_, 'variant': v, 'rule': rule, 'f_R0m': fm, **t, 'vs_R0_calmar_diff': t0['calmar_diff'], 'vs_R0_lo95': t0['lo95'], 'vs_R0_hi95': t0['hi95'], 'vs_R0_p': t0['p']})
    print(f'f = {f_:.2%} done, {time.time() - T0:.0f}s')
DISC = pd.concat(DISC_ALL, ignore_index=True); EVAL = pd.DataFrame(EVAL_ALL); TESTS = pd.DataFrame(TEST_ALL)
TESTS['p_holm'] = np.nan
for f_ in FS:
    m = TESTS.f == f_; TESTS.loc[m, 'p_holm'] = tp.holm(TESTS.loc[m, 'p'].to_numpy())
TESTS['result'] = np.where(TESTS.p_holm < PP0.alpha, 'IMPROVES', 'NO_DIFFERENCE')
print('selected H, C:', {f'{k[0]:.2%} {k[1]}': v for k, v in SEL_ALL.items()})
cols = ['f', 'variant', 'rule', 'cagr', 'max_dd', 'calmar', 'sharpe', 'ruined', 'max_open_risk', 'mean_open_risk', 'max_gross_lev', 'entries', 'skipped', 'scaled']
print('Discovery'); display(DISC[cols[:2] + ['rule', 'H', 'C'] + cols[3:]].round(3))
print('Evaluation'); display(EVAL[cols[:3] + ['param'] + cols[3:]].round(3))
display(TESTS.round(4))

## 3. Decision (f = 0.5%)

In [ ]:
TP = TESTS[TESTS.f == F_PRIMARY]
DEC = {'f': F_PRIMARY, 'selected': {v: SEL_ALL[(F_PRIMARY, v)] for v in ('long', 'long_rev')},
       'tests': TP[['variant', 'rule', 'calmar_diff', 'lo95', 'hi95', 'p', 'p_holm', 'result']].to_dict('records'),
       'any_ruined_in_evaluation': bool(EVAL[(EVAL.f == F_PRIMARY)].ruined.any())}
imp = [f'{r.rule}/{r.variant}' for r in TP.itertuples() if r.result == 'IMPROVES']
DEC['verdict'] = 'RULE_IMPROVES: ' + ', '.join(imp) if imp else 'NO_RULE_BEATS_SAME_EXPOSURE'
print(json.dumps(DEC, indent=1, default=float))

## 4. Reported: reverse vs long, sleeves, full period, 4-coin portfolio (f = 0.5%)

In [ ]:
pp = replace(PP0, f=F_PRIMARY)
rows = []
for f_ in FS:
    for rule in ('R0', 'R1', 'R2', 'R3', 'R4'):
        t = tp.calmar_diff(CURVES[(f_, 'long_rev', rule)], CURVES[(f_, 'long', rule)], pp, pp.seed + 500)
        e = EVAL.set_index(['f', 'variant', 'rule'])
        rows.append({'f': f_, 'rule': rule, 'calmar_long': e.loc[(f_, 'long', rule), 'calmar'], 'calmar_long_rev': e.loc[(f_, 'long_rev', rule), 'calmar'],
                     'dd_long': e.loc[(f_, 'long', rule), 'max_dd'], 'dd_long_rev': e.loc[(f_, 'long_rev', rule), 'max_dd'], **t})
REV = pd.DataFrame(rows); print('Evaluation: long + reverse minus long (Calmar)'); display(REV.round(3))

SLV = []
pps = tp.PParams()            # TB 10 sleeves: each coin its own account, 2% of its HWM
for v in ('long', 'long_rev'):
    for name, (a0, a1) in (('discovery', (DISC0, DISC1)), ('evaluation', (EVAL0, None))):
        cur = {}
        for a, bk in BOOKS[v].items():
            o = tp.simulate({a: bk}, 'R0', pps, a0, a1)
            if len(o['equity']) > 30 and o['stats']['entries'] > 0:
                cur[a] = tp.daily_curve(o['equity'], pps.start_equity)
        lr = ts.pooled_log_returns(cur); Dd = pps.start_equity * np.exp(lr.cumsum())
        SLV.append({'variant': v, 'window': name, 'coins': len(cur), **tp.metrics(Dd, pps.start_equity)})
        if name == 'evaluation':
            CURVES[(F_PRIMARY, v, 'sleeves')] = Dd
SLV = pd.DataFrame(SLV); print('Equal-weight sleeves'); display(SLV.round(3))

FULL = []
for v in ('long', 'long_rev'):
    sel = SEL_ALL[(F_PRIMARY, v)]
    for rule, kw in (('R0', {}), ('R1', {'H': sel['H']}), ('R2', {}), ('R3', {'C': sel['C']}), ('R4', {})):
        m, _, _ = run(v, rule, DISC0, None, pp, **kw); FULL.append({'variant': v, 'rule': rule, **m})
        if not SYNTHETIC:
            core = {a: BOOKS[v][a] for a in tf.CONFIRM_ASSETS if a in BOOKS[v]}
            m4, _, _ = run(v, rule, tb.COMMON_START, None, pp, books=core, **kw); FULL.append({'variant': v, 'rule': rule + '_4coins', **m4})
FULL = pd.DataFrame(FULL); print('Full period and 4-coin portfolio, f = 0.5%')
display(FULL[cols[1:]].round(3))

## 5. Figures (f = 0.5%)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
colors = {'R0': 'k', 'R1': '#1f5aa6', 'R2': '#3a7d44', 'R3': '#c2571a', 'R4': '#8a4fa3', 'sleeves': '0.55'}
for j, v in enumerate(('long', 'long_rev')):
    for rule in ('R0', 'R1', 'R2', 'R3', 'R4', 'sleeves'):
        c = CURVES[(F_PRIMARY, v, rule)]
        axes[0, j].plot(c.index, c / c.iloc[0], color=colors[rule], lw=1.3 if rule != 'sleeves' else 1.0, ls='-' if rule != 'sleeves' else '--', label=rule)
        axes[1, j].plot(c.index, (c / c.cummax() - 1) * 100, color=colors[rule], lw=1.0, ls='-' if rule != 'sleeves' else '--', label=rule)
    axes[0, j].set_title(f'Evaluation equity, {v}, f = 0.5%', loc='left', fontsize=10); axes[1, j].set_title(f'Drawdown %, {v}', loc='left', fontsize=10)
    for a in axes[:, j]:
        a.grid(alpha=0.25); a.legend(frameon=False, fontsize=8, ncol=3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(14, 3.5))
for rule in ('R0', 'R1', 'R2', 'R3'):
    s = OUTS[(F_PRIMARY, 'long', rule)]['open_risk']; ax.plot(s.index, s * 100, color=colors[rule], lw=0.9, label=rule)
ax.set_ylabel('open risk, % of equity'); ax.set_title('Evaluation, long only, f = 0.5%: open risk', loc='left', fontsize=10); ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
for (v, mk) in (('long', 'o'), ('long_rev', 's')):
    for rule in ('R0', 'R1', 'R2', 'R3', 'R4'):
        e = EVAL[(EVAL.variant == v) & (EVAL.rule == rule)]
        ax.plot(e.f * 100, e.calmar, marker=mk, color=colors[rule], ls='-' if v == 'long' else ':', label=f'{rule} {v}')
ax.set_xlabel('base risk per trade f (%)'); ax.set_ylabel('evaluation Calmar'); ax.legend(frameon=False, fontsize=7, ncol=2); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
DISC.to_csv(OUT / '12_discovery.csv', index=False); EVAL.to_csv(OUT / '12_evaluation.csv', index=False); TESTS.to_csv(OUT / '12_tests.csv', index=False)
REV.to_csv(OUT / '12_reverse_vs_long.csv', index=False); SLV.to_csv(OUT / '12_sleeves.csv', index=False); FULL.to_csv(OUT / '12_full_and_4coins.csv', index=False)
COV.to_csv(OUT / '12_coverage.csv', index=False)
pd.DataFrame({f'{k[0]}|{k[1]}|{k[2]}': c for k, c in CURVES.items()}).to_csv(OUT / '12_eval_curves.csv.gz', compression='gzip')
(OUT / '12_decision.json').write_text(json.dumps({'decision': DEC, 'registration': REG, 'synthetic': SYNTHETIC}, indent=1, default=str))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '12_manifest.json').write_text(json.dumps({'version': tp.VERSION, 'git': git, 'python': platform.python_version(),
    'created_utc': pd.Timestamp.now('UTC').isoformat(), 'coins': list(BOOKS['long']), 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print('VERDICT:', DEC['verdict'])